## Experiment 4

We test the hypothesis that sentence patterns come from local patterns rather than a meaningful direction. For each direction, we will take the 10 top-activating sentences and compare each one of them against its 10 nearest neighbors and the other 9 top-activating sentences.

In [10]:
import numpy as np
import pandas as pd
import glob
import os
import json

# dataset names
ds_names = ['qqp', 'qnli', 'wiki', 'books']

# load embeddings for each of the 4 datasets
embeds = { name: np.load(f'../../../embeds/full/{name}_embed.npy', mmap_mode='r') 
           for name in ds_names }

# load annotated panels
panels = json.load(open('../../../panels/panels_dedup.json'))

# collect top sentences for neurons and directions here
# we already had the panel files, so we can just read those again
dirs = []

# paths of each panel file for neurons and directions
# neurons are also "directions", except they have one 1 component and all others 0
# the other random directions have all components equal to some random value
paths = (glob.glob('../../../panels/parquet/*_neur_*.parquet') + # neuron .parquets
         glob.glob('../../../panels/parquet/*_dir_*.parquet'))   # direction .parquets


for path in paths:
    # file format is "dataset_condition_index.parquet"
    dataset, condition, idx = (os.path.basename(path)
                               .replace('.parquet', '') # remove the extension
                               .split('_'))             # split by underline to get components
    
    # collect direction data
    dirs.append({'file': os.path.basename(path), 
                 'dataset': dataset, 
                 'condition': condition,
                 'idx': int(idx), # was a string from the .split(), need to convert
                 'top': pd.read_parquet(path)['id'].to_numpy()
                })

We implement the locality metric given by the authors.

In [11]:
# Jaccard-like locality metric as defined by the paper
def loc(a, b):
    space = np.linspace(min(a.min(), b.min()), 
                        max(a.max(), b.max()), 
                        101)                   # binning count does matter 
    h1 = np.histogram(a, space)[0] / len(a)    # value of h1
    h2 = np.histogram(b, space)[0] / len(b)    # value of h2
    return np.sum(np.minimum(h1, h2)) / np.sum(np.maximum(h1, h2))


# mean locality per sentence
def loc_ps(nn, gram):
    # nn[i][j]   = similarity of i-th top-activating sentence with its j-th nearest neighbor
    # gram[i][j] = similarity of i-th top-activating sentence with j-th top activating sentence
    #              this is a matrix of dot products between 10 items = a Gram matrix
    return np.mean([loc(nn[k], np.delete(gram[k], k)) # delete similarity with self here
                    for k in range(10)])

In [22]:
from scipy.stats import mannwhitneyu

# seeded rng
rng = np.random.default_rng(1)

# construct the table here
tab = []

# go once for each dataset
for name in ds_names:
    # get the sentence embeddings for this dataset
    embed = np.array(embeds[name])

    # dirs collected top-activating sentences from all the files, across all datasets
    # we pass through each dataset in order so we filter out only the data for the current set
    ds_dirs = filter(lambda d: d['dataset'] == name, dirs)
    # for each set of top-activating sentences:
    # (be they from a neuron-direction or a random direction)
    for direction in ds_dirs:

        # index the embed variable to get embeds for top-activating sentences
        dir_embed = embed[direction['top']]
        # get sentence similarities for top sentences, shape (10, dataset size)
        ss = dir_embed @ embed.T
        # dot products between the top 10 sentences, resulting shape is (10, 10)
        gram = dir_embed @ dir_embed.T

        # if we look at a triangle of this matrix (which is symmetrical)
        # we obtain the similarity scores between pairs of top sentences
        top_pairs = gram[np.triu_indices(10, k=1)]

        # use a copy
        ss_copy = ss.copy()
        np.put_along_axis(ss_copy, direction['top'][:, None], -np.inf, axis=1)
        # find the top 10 nearest neighbors
        nn = np.sort(ss_copy, axis=1)[:, -10:]

        # also take 10 random sentences
        random = np.take_along_axis(ss, rng.choice(len(embed), (10, 10)), axis=1)
        random = random.ravel()

        meaningful = False
        # if we had annotations for the panel for this direction, it is meaningful
        if len(panels.get(direction['file'].replace('.parquet', '.csv'), [])) > 0:
            meaningful = True

        # add entry to the table
        tab.append({'dataset': name, 
                    'condition': direction['condition'], 
                    'idx': direction['idx'],
                    'meaningful': meaningful,
                    'nn': nn.ravel(),
                    'top_pairs': top_pairs, 
                    'random': random,
                    'locality': loc_ps(nn, gram)})


# result
res = pd.DataFrame(tab)
res

,dataset,condition,idx,meaningful,nn,top_pairs,random,locality
0,qqp,neur,206,False,"[200.74821, 200.89078, 201.61572, 202.38066, 2...","[177.87216, 199.46086, 192.23166, 173.27957, 1...","[158.43718, 167.10815, 187.56628, 178.57133, 1...",0.005882
1,qqp,neur,661,False,"[174.49284, 174.58157, 175.50276, 176.16321, 1...","[132.31885, 127.35767, 145.00267, 146.90909, 1...","[134.56787, 142.01709, 139.07368, 152.36148, 1...",0.005263
2,qqp,neur,563,True,"[215.12802, 215.19815, 215.30089, 215.79568, 2...","[211.89026, 180.9187, 124.553535, 157.79391, 1...","[165.27371, 156.40475, 167.03842, 189.97011, 1...",0.017028
3,qqp,neur,107,True,"[147.46661, 147.48364, 147.53896, 147.79904, 1...","[134.64581, 140.90582, 121.7359, 132.95102, 13...","[111.07361, 117.11702, 121.97832, 115.74516, 1...",0.000000
4,qqp,neur,655,False,"[213.82635, 214.22372, 214.37646, 215.19781, 2...","[172.04625, 185.88992, 190.62326, 186.76425, 1...","[171.8486, 155.56482, 175.74835, 175.84637, 16...",0.000000
...,...,...,...,...,...,...,...,...
227,books,dir,16,True,"[198.12, 198.23083, 198.32828, 198.4216, 198.5...","[188.0589, 182.62979, 177.40993, 175.86514, 18...","[131.25539, 170.0107, 125.434715, 117.77646, 1...",0.005263
228,books,dir,5,True,"[186.95045, 187.09818, 188.00073, 188.0186, 18...","[152.3496, 154.89742, 177.26114, 163.04257, 16...","[159.48914, 143.70947, 141.99762, 126.32885, 1...",0.005882
229,books,dir,23,True,"[206.84108, 206.87465, 206.92096, 206.94714, 2...","[193.50317, 199.88872, 186.10744, 192.34666, 1...","[166.79944, 172.54521, 185.15991, 167.28221, 1...",0.000000
230,books,dir,15,True,"[207.42166, 207.54547, 212.28369, 213.8208, 21...","[231.84445, 207.54555, 195.72803, 194.44054, 1...","[39.243664, 45.25949, 39.878304, 40.828598, 32...",0.064244


Now that we have all the scores, we can compare and contrast the scores of meaningful and meaningless neurons, as identified through annotation.

In [23]:
tab = []

# construct table 5
for ds_name in ['all'] + ds_names:
    # for each datset, only use matching rows of the result
    # or all of them, if the dataset is "all of them"
    rows = None
    if ds_name == 'all':
        # handle separately, since it's not an actual 
        # dataset name in the collected result
        rows = res
    else:
        matching = res['dataset'] == ds_name
        rows = res[matching]

    # locality scores of meaningful (1) directions
    m1 = rows[rows['meaningful']]['locality']
    # locality scores of meaningless (0) directions
    m0 = rows[~rows['meaningful']]['locality']

    tab.append({
        'dataset':     ds_name,
        'meaningful':  m1.mean(),
        'meaningless': m0.mean(),
        'p':           mannwhitneyu(m0, m1).pvalue,
    })


# show it
df = pd.DataFrame(tab)
df.round(4)

,dataset,meaningful,meaningless,p
0,all,0.0433,0.0101,0.0004
1,qqp,0.0148,0.0040,0.0147
2,qnli,0.0410,0.0113,0.4760
3,wiki,0.0488,0.0189,0.3037
4,books,0.0629,0.0226,0.4071
